# Train or resume DeBERTa and the bubble narration classifier
Use **Godfather**. Select either model or both in the first cell. `EPOCHS` is the total target: 10 → 40 continues with epochs 11–40. Change EPOCHS and rerun the training cell, or run all cells after restarting the kernel.

`last.pt` stores both models, optimizer/scaler, sampler/random states, history and best score. `best.pt` stores the best pair, including epoch 0. The best encoder, classifier and individual layer files are exported automatically at RUN root without resetting live training weights. Existing experiment folders are preserved.

Shared run paths live in config.py under TRAINING_PATHS. Notebook 07 writes deberta_run; notebook 05 reads that source and writes classifier_run; notebook 06 evaluates either run explicitly.


## 1. Training switches, initialization checkpoints and total epochs
UseClassifierCheckpoint=False starts a random classifier. DEBERTA_CHECKPOINT_PATH accepts a Hugging Face folder, joint best.pt/last.pt, or raw encoder state dict. DEBERTA_BASE_PATH supplies config/tokenizer for .pt files. Layer dictionaries optionally override individual zero-based DeBERTa layers or named classifier modules. An override file can contain that module alone or a complete model checkpoint. ResumeTraining=True prioritizes this run’s last.pt over initialization paths. Change RUN for different train switches or input settings.

Full and layer initialization paths apply to new runs. ResumeTraining=True loads the run’s complete last.pt instead; choose a new RUN to mix different checkpoints.

Checkpoint inputs are independent: set DEBERTA_CHECKPOINT_PATH to a full encoder folder or a joint checkpoint file; optionally fill DEBERTA_LAYER_CHECKPOINTS with layer-index/file pairs. CLASSIFIER_CHECKPOINT_PATH controls full classifier loading, while CLASSIFIER_LAYER_CHECKPOINTS overrides named modules. Layer indices are zero-based.


In [ ]:
from pathlib import Path
import sys
import json
import random
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, WeightedRandomSampler

ROOT = Path(r'C:/Users/ESHAAN/BYTE/Applied-Task/Magi/FINETUNING-EXPERMENTS/NarrationModelwBubbles')
sys.path.insert(0, str(ROOT))
for name in ['config', 'model', 'bubble_features', 'training_state', 'checkpoint_inputs']:
    sys.modules.pop(name, None)
from config import CONFIG, TRAINING_PATHS, fingerprint, save_json
from model import BoxClassifier, collate, metrics, prediction_rows
from bubble_features import sha256, model_files
from training_state import load_resume, restore_state, save_progress
from checkpoint_inputs import prepare_encoder, frozen_samples, read_weights, load_layer, checkpoint_identity, export_best, check_training_inputs

from transformers import AutoModel, AutoTokenizer

TrainDeberta = True
TrainClassifier = True
UseClassifierCheckpoint = False
CLASSIFIER_CHECKPOINT_PATH = TRAINING_PATHS['deberta_run'] / 'classifier.pt'
DEBERTA_BASE_PATH = Path(CONFIG['deberta_model'])
DEBERTA_CHECKPOINT_PATH = DEBERTA_BASE_PATH  # Hugging Face folder, best.pt, last.pt or raw encoder weights.
DEBERTA_LAYER_CHECKPOINTS = {}  # Example: {11: Path(r'C:/my_run/deberta_layers/layer_11.pt')}
CLASSIFIER_LAYER_CHECKPOINTS = {}  # Example: {'classifier': Path(r'C:/my_run/classifier_layers/classifier.pt')}
ResumeTraining = True
RUN = TRAINING_PATHS['deberta_run']
EPOCHS = 10
UNFREEZE_LAYERS = 2
ENCODER_LR = 1e-5
CLASSIFIER_LR = 1e-4
TEXT_CHUNK_SIZE = 2
ACCUMULATION_STEPS = 4

assert torch.cuda.is_available(), 'Select the Godfather CUDA kernel.'
assert TEXT_CHUNK_SIZE > 0 and ACCUMULATION_STEPS > 0
DEVICE = 'cuda'
AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
scaler = torch.amp.GradScaler('cuda', enabled=AMP_DTYPE == torch.float16)

random.seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])
torch.manual_seed(CONFIG['seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG['seed'])
assert isinstance(EPOCHS, int) and EPOCHS >= 0
saved = load_resume(RUN, ResumeTraining)
print('Resume epoch:', saved['epoch'] if saved else 0, '| target epochs:', EPOCHS)

ENCODER_INPUT_PATH = Path(saved.get('encoder_source', DEBERTA_CHECKPOINT_PATH)) if saved else DEBERTA_CHECKPOINT_PATH
check_training_inputs(DEBERTA_BASE_PATH, ENCODER_INPUT_PATH, CLASSIFIER_CHECKPOINT_PATH,
                      UseClassifierCheckpoint, DEBERTA_LAYER_CHECKPOINTS, CLASSIFIER_LAYER_CHECKPOINTS, saved)
print('Output run:', RUN)


Inspect checkpoint contents.


In [14]:
ENCODER_PATH = prepare_encoder(DEBERTA_BASE_PATH, ENCODER_INPUT_PATH,
                               {} if saved else DEBERTA_LAYER_CHECKPOINTS, CONFIG['cache_dir'])
SAMPLES_PATH = CONFIG['cache_dir'] / 'samples.pt'
if not SAMPLES_PATH.is_file():
    raise FileNotFoundError('Prepare the bubble training caches before running this notebook.')
data = torch.load(SAMPLES_PATH, map_location='cpu', weights_only=True)
samples = data['samples']
train = [samples[i] for i in data['train_indices'] if (samples[i]['labels'] != -100).any()]
val = [samples[i] for i in data['val_indices']]
train_ids = {sample['sequence_id'] for sample in train}
val_ids = {sample['sequence_id'] for sample in val}
assert not train_ids & val_ids
train_pages = {page['sha256'] for sample in train for page in sample['pages']}
val_pages = {page['sha256'] for sample in val for page in sample['pages']}
assert not train_pages & val_pages
assert all('ocr_selection' in box for sample in samples for box in sample['boxes'])

tokenizer = AutoTokenizer.from_pretrained(ENCODER_PATH, local_files_only=True)
tokenizer.padding_side = 'right'
token_signature = fingerprint({
    'samples': data['signature'],
    'deberta_files': model_files(ENCODER_PATH),
    'max_tokens': CONFIG['max_tokens'],
    'context': 2,
    'pooling': 'target_mean_v1',
    'tokenization_version': 1,
})
TOKEN_CACHE = CONFIG['cache_dir'] / 'deberta_finetune_tokens.pt'
token_cache = torch.load(TOKEN_CACHE, weights_only=True) if TOKEN_CACHE.exists() else None
if token_cache is None or token_cache['signature'] != token_signature:
    tokens = {}
    for sample in samples:
        history, items = {}, []
        for box in sample['boxes']:
            previous = history.setdefault(box['page_index'], [])
            context = []
            for text in previous[-2:]:
                context += tokenizer.encode(text, add_special_tokens=False) + [tokenizer.sep_token_id]
            target = tokenizer.encode(box['ocr'], add_special_tokens=False) or [tokenizer.unk_token_id]
            target = target[-(CONFIG['max_tokens'] - 2):]
            remaining = CONFIG['max_tokens'] - 2 - len(target)
            context = context[-remaining:] if remaining else []
            ids = [tokenizer.cls_token_id] + context + target + [tokenizer.sep_token_id]
            target_mask = [0] * (1 + len(context)) + [1] * len(target) + [0]
            items.append({'box_id': box['box_id'], 'ids': ids, 'target_mask': target_mask})
            previous.append(box['ocr'])
        tokens[sample['sequence_id']] = items
    token_cache = {'signature': token_signature, 'tokens': tokens}
    torch.save(token_cache, TOKEN_CACHE)
tokens = token_cache['tokens']
print('Sequences: train', len(train), '| validation', len(val))
print('Original train label counts:', data['train_counts'].tolist())
print('Class weights:', data['class_weights'].tolist())


c:\Users\ESHAAN\BYTE\godfather.venv\Lib\site-packages\transformers\convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
The tokenizer you are loading from 'C:\MODELS\deberta-v3-base' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Sequences: train 63 | validation 16
Original train label counts: [1142, 50]
Class weights: [0.13372045755386353, 1.8662796020507812]


## 3. Load checkpoints and apply the train switches
Only enabled parameters enter the optimizer. Frozen models use evaluation mode. The last checkpoint restores both weights and optimizer in the training cell.


In [15]:
deberta = AutoModel.from_pretrained(ENCODER_PATH, local_files_only=True).to(DEVICE)
assert deberta.config.hidden_size == 768
layers = deberta.encoder.layer
assert 1 <= UNFREEZE_LAYERS <= len(layers)
deberta.requires_grad_(False)
if TrainDeberta:
    for layer in layers[-UNFREEZE_LAYERS:]:
        layer.requires_grad_(True)
    deberta.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})

visual_dim = train[0]['visual'].shape[-1]
model_args = {key: CONFIG[key] for key in ['dropout', 'num_heads', 'dim_feedforward']}
initial = saved
classifier_weights = saved['state_dict'] if saved is not None else None
if saved is None and UseClassifierCheckpoint:
    classifier_weights, initial = read_weights(CLASSIFIER_CHECKPOINT_PATH, 'classifier')
if initial:
    assert initial.get('visual_dim', visual_dim) == visual_dim
    assert initial.get('max_positions', data['max_positions']) == data['max_positions']
    assert initial.get('feature_contract', {}).get('name', 'NarrationModelwBubbles') == 'NarrationModelwBubbles'
    model_args.update(initial.get('model_args', {}))
classifier = BoxClassifier(visual_dim, data['max_positions'], **model_args).to(DEVICE)
if classifier_weights is not None:
    classifier.load_state_dict(classifier_weights, strict=True)
if saved is None:
    for module_name, path in CLASSIFIER_LAYER_CHECKPOINTS.items():
        load_layer(classifier.get_submodule(module_name), path, module_name, 'classifier')
classifier.requires_grad_(TrainClassifier)

encoder_parameters = [parameter for parameter in deberta.parameters() if parameter.requires_grad]
classifier_parameters = [parameter for parameter in classifier.parameters() if parameter.requires_grad]
groups = []
if encoder_parameters:
    groups.append({'params': encoder_parameters, 'lr': ENCODER_LR})
if classifier_parameters:
    groups.append({'params': classifier_parameters, 'lr': CLASSIFIER_LR})
optimizer = torch.optim.AdamW(groups, weight_decay=CONFIG['weight_decay']) if groups else None
trainable_parameters = encoder_parameters + classifier_parameters
models = {'classifier': classifier, 'deberta': deberta}
criterion = nn.CrossEntropyLoss(weight=data['class_weights'].to(DEVICE), ignore_index=-100, reduction='sum')
weights = [CONFIG['narration_sampling_weight'] if (sample['labels'] == 1).any() else 1.0 for sample in train]
sampler = WeightedRandomSampler(weights, len(train), replacement=True,
                                generator=torch.Generator().manual_seed(CONFIG['seed']))
train_loader = DataLoader(train, batch_size=1, sampler=sampler, collate_fn=lambda rows: rows[0])
val_loader = DataLoader(val, batch_size=1, shuffle=False, collate_fn=lambda rows: rows[0])

feature_contract = {
    'name': 'NarrationModelwBubbles', 'settings': CONFIG['bubble_settings'],
    'helper_sha256': sha256(ROOT / 'bubble_features.py'),
    'ocr_selector_sha256': sha256(ROOT / 'ocr_selection.py'),
    'bubble_model_sha256': sha256(CONFIG['bubble_settings']['bubble_model']),
    'magi_files': model_files(Path(CONFIG['magi_model'])),
    'deberta_files': model_files(ENCODER_PATH),
    'max_tokens': CONFIG['max_tokens'],
}

contract = fingerprint({
    'samples': data['signature'], 'tokens': token_signature, 'features': feature_contract,
    'model_args': model_args, 'train_classifier': TrainClassifier, 'train_deberta': TrainDeberta,
    'unfreeze_layers': UNFREEZE_LAYERS, 'encoder_lr': ENCODER_LR, 'classifier_lr': CLASSIFIER_LR,
    'accumulation_steps': ACCUMULATION_STEPS, 'text_chunk_size': TEXT_CHUNK_SIZE,
    'amp_dtype': str(AMP_DTYPE), 'weight_decay': CONFIG['weight_decay'],
    'sampling': CONFIG['narration_sampling_weight'], 'seed': CONFIG['seed'],
    'code': sha256(ROOT / 'training_state.py'),
})
if CLASSIFIER_LAYER_CHECKPOINTS:
    contract = fingerprint({'base_contract': contract, 'classifier_layers': {
        key: checkpoint_identity(path) for key, path in CLASSIFIER_LAYER_CHECKPOINTS.items()}})
metadata = {
    'training_contract': contract, 'samples_signature': data['signature'],
    'visual_dim': visual_dim, 'max_positions': data['max_positions'],
    'model_args': model_args, 'class_weights': data['class_weights'],
    'feature_contract': feature_contract,
    'encoder_source': str(ENCODER_PATH),
    'finetuning_settings': {'unfreeze_layers': UNFREEZE_LAYERS,
                          'train_deberta': TrainDeberta, 'train_classifier': TrainClassifier},
}
print('Trainable encoder parameters:', sum(parameter.numel() for parameter in encoder_parameters))
print('Trainable classifier parameters:', sum(parameter.numel() for parameter in classifier_parameters))

del initial, classifier_weights


Trainable encoder parameters: 14175744
Trainable classifier parameters: 1215442


## 4. Live embeddings, training and validation
Gradients pass through DeBERTa only when enabled. A frozen classifier still passes gradients to a trainable encoder. Accumulation preserves complete sequences and normalizes by labeled-box count.


In [16]:
def set_model_mode(training):
    deberta.train(training and TrainDeberta)
    classifier.train(training and TrainClassifier)
    if training and TrainDeberta:
        deberta.embeddings.eval()
        for layer in layers[:-UNFREEZE_LAYERS]:
            layer.eval()


def live_text_features(sample):
    items = tokens[sample['sequence_id']]
    assert [item['box_id'] for item in items] == [box['box_id'] for box in sample['boxes']]
    vectors = []
    for start in range(0, len(items), TEXT_CHUNK_SIZE):
        chunk = items[start:start + TEXT_CHUNK_SIZE]
        length = max(len(item['ids']) for item in chunk)
        ids = torch.full((len(chunk), length), tokenizer.pad_token_id, dtype=torch.long, device=DEVICE)
        attention = torch.zeros_like(ids)
        target_mask = torch.zeros_like(ids, dtype=torch.float)
        for i, item in enumerate(chunk):
            n = len(item['ids'])
            ids[i, :n] = torch.tensor(item['ids'], device=DEVICE)
            attention[i, :n] = 1
            target_mask[i, :n] = torch.tensor(item['target_mask'], device=DEVICE)
        hidden = deberta(input_ids=ids, attention_mask=attention).last_hidden_state
        pooled = (hidden.float() * target_mask[..., None]).sum(1) / target_mask.sum(1, keepdim=True)
        vectors.append(pooled)
    return torch.cat(vectors, dim=0)


def gradient_norm(parameters):
    squares = [parameter.grad.detach().float().square().sum()
               for parameter in parameters if parameter.grad is not None]
    return float(torch.stack(squares).sum().sqrt()) if squares else 0.0


def run_epoch(loader, training):
    set_model_mode(training)
    loss_sum, labeled_count, rows = 0.0, 0, []
    encoder_norms, classifier_norms = [], []
    group_count = 0
    if training:
        optimizer.zero_grad(set_to_none=True)
    for step, sample in enumerate(loader, 1):
        batch = collate([sample])
        batch = {key: value.to(DEVICE) if torch.is_tensor(value) else value for key, value in batch.items()}
        count = int((batch['labels'] != -100).sum())
        with torch.set_grad_enabled(training):
            with torch.autocast('cuda', dtype=AMP_DTYPE):
                batch['text'] = live_text_features(sample)[None]
                logits = classifier(batch)
                summed = criterion(logits.float().reshape(-1, 2), batch['labels'].reshape(-1))
            if not torch.isfinite(summed):
                raise FloatingPointError('Nonfinite narration loss; stop and inspect this run.')
            if training:
                scaler.scale(summed).backward()
                group_count += count
                if step % ACCUMULATION_STEPS == 0 or step == len(loader):
                    scaler.unscale_(optimizer)
                    for parameter in trainable_parameters:
                        if parameter.grad is not None:
                            parameter.grad.div_(max(group_count, 1))
                    encoder_norms.append(gradient_norm(encoder_parameters))
                    classifier_norms.append(gradient_norm(classifier_parameters))
                    torch.nn.utils.clip_grad_norm_(trainable_parameters, 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)
                    group_count = 0
        loss_sum += float(summed.detach())
        labeled_count += count
        rows.extend(prediction_rows(batch, logits.detach().float().softmax(-1).cpu()))
        if step % 16 == 0 or step == len(loader):
            print(('train' if training else 'val'), f'{step}/{len(loader)} sequences', flush=True)
    result = metrics(rows)
    result['loss'] = loss_sum / labeled_count if labeled_count else None
    if encoder_norms:
        result['encoder_grad_norm'] = float(np.mean(encoder_norms))
        result['classifier_grad_norm'] = float(np.mean(classifier_norms))
    return result, rows


## 5. Train to the total EPOCHS target
Change EPOCHS and rerun this cell to extend training. Existing history and best score are restored. All epochs save a resumable snapshot; epoch 0 protects the starting pair from a worse fine-tuned result.


In [17]:
# Reload last.pt every time this cell runs, including after a kernel restart.
saved = load_resume(RUN, ResumeTraining)
if TrainClassifier or TrainDeberta:
    restore_state(saved, models, optimizer, scaler, sampler, contract)
elif saved is not None:
    assert saved['samples_signature'] == data['signature'], 'Evaluation inputs changed.'
    for name, model in models.items():
        model.load_state_dict(saved['models'][name], strict=True)
history = list(saved['history']) if saved else []
completed = saved['epoch'] if saved else 0
best_f1 = saved['best_f1'] if saved else -1.0
best_epoch = saved['best_epoch'] if saved else 0

if not (TrainClassifier or TrainDeberta):
    validation, _ = run_epoch(val_loader, False)
    print('Evaluation only:', json.dumps(validation, indent=2))
else:
    if saved is None:
        baseline, rows = run_epoch(val_loader, False)
        best_f1 = baseline['macro_f1']
        metadata['baseline_metrics'] = baseline
        save_progress(RUN, models, optimizer, scaler, sampler, metadata,
                      0, history, best_f1, 0, baseline, True)
        save_json(RUN / 'best_validation_predictions.json', rows)
        export_best(RUN, models, metadata, ENCODER_PATH, tokenizer)
        print('Epoch 0 validation:', json.dumps(baseline, indent=2))
    else:
        metadata['baseline_metrics'] = saved['baseline_metrics']
    saved = None
    for epoch in range(completed + 1, EPOCHS + 1):
        print(f'Epoch {epoch}/{EPOCHS}', flush=True)
        train_metrics, _ = run_epoch(train_loader, True)
        val_metrics, rows = run_epoch(val_loader, False)
        history.append({'epoch': epoch, 'train': train_metrics, 'val': val_metrics})
        improved = val_metrics['macro_f1'] > best_f1
        if improved:
            best_f1, best_epoch = val_metrics['macro_f1'], epoch
        save_progress(RUN, models, optimizer, scaler, sampler, metadata,
                      epoch, history, best_f1, best_epoch, val_metrics, improved)
        save_json(RUN / f'val_predictions_epoch_{epoch:02d}.json', rows)
        if improved:
            save_json(RUN / 'best_validation_predictions.json', rows)
            export_best(RUN, models, metadata, ENCODER_PATH, tokenizer)
        completed = epoch
        print(json.dumps(history[-1], indent=2), flush=True)
    print('Completed:', completed, '| best epoch:', best_epoch, '| best macro F1:', best_f1)
    print('Increase EPOCHS and rerun this cell to continue.')


val 16/16 sequences
Epoch 0 validation: {
  "macro_f1": 0.4673750703914991,
  "balanced_accuracy": 0.5048582995951417,
  "SPOKEN": {
    "precision": 0.9568965517241379,
    "recall": 0.7789473684210526,
    "f1": 0.8588007736943907,
    "support": 285,
    "p_true_mean": 0.6227384609088563,
    "p_true_median": 0.6263527274131775,
    "p_true_gt_0.9": 0.021052631578947368,
    "p_true_lt_0.5": 0.22105263157894736
  },
  "NARRATED": {
    "precision": 0.045454545454545456,
    "recall": 0.23076923076923078,
    "f1": 0.0759493670886076,
    "support": 13,
    "p_true_mean": 0.32488806316485774,
    "p_true_median": 0.244723841547966,
    "p_true_gt_0.9": 0.0,
    "p_true_lt_0.5": 0.7692307692307693
  },
  "loss": 0.16774121386892843
}
Epoch 1/10
train 16/63 sequences
train 32/63 sequences
train 48/63 sequences
train 63/63 sequences
val 16/16 sequences
{
  "epoch": 1,
  "train": {
    "macro_f1": 0.3741922704190943,
    "balanced_accuracy": 0.4848316210045662,
    "SPOKEN": {
      "pre

## 6. Inspect validation history


In [18]:
import pandas as pd

if (RUN / 'last.pt').is_file():
    progress = torch.load(RUN / 'last.pt', map_location='cpu', weights_only=True)
    display(pd.DataFrame([
        {'epoch': row['epoch'], 'train_loss': row['train']['loss'], 'val_loss': row['val']['loss'],
         'macro_f1': row['val']['macro_f1'],
         'narration_precision': row['val']['NARRATED']['precision'],
         'narration_recall': row['val']['NARRATED']['recall'],
         'narration_f1': row['val']['NARRATED']['f1']}
        for row in progress['history']
    ]))
    print('Best epoch:', progress['best_epoch'], '| best macro F1:', progress['best_f1'])
    del progress


,epoch,train_loss,val_loss,macro_f1,narration_precision,narration_recall,narration_f1
0,1,0.266413,0.171169,0.232236,0.047826,0.846154,0.090535
1,2,0.179047,0.146001,0.476274,0.000000,0.000000,0.000000
2,3,0.161312,0.125091,0.595783,0.176471,0.461538,0.255319
3,4,0.113071,0.136970,0.486207,0.000000,0.000000,0.000000
4,5,0.116658,0.101491,0.758704,0.538462,0.538462,0.538462
5,6,0.080945,0.066095,0.718921,0.360000,0.692308,0.473684
6,7,0.038236,0.095740,0.737676,0.466667,0.538462,0.500000
7,8,0.020926,0.104228,0.747836,0.500000,0.538462,0.518519
8,9,0.016167,0.251873,0.678879,1.000000,0.230769,0.375000
9,10,0.007953,0.198407,0.713572,0.800000,0.307692,0.444444


Best epoch: 5 | best macro F1: 0.7587044534412956


## 7. Full inference paths
Every improvement saves RUN/best.pt, RUN/deberta/, RUN/inference_config.py, RUN/classifier.pt and individual model layer files. Copy the printed paths into full inference. No epoch-specific export folder is used.


In [19]:
print('Best model and inference export are saved automatically at the run root.')
print('NARRATION_MODEL_PATH =', repr(str(ROOT / 'model.py')))
print('NARRATION_CHECKPOINT_PATH =', repr(str(RUN / 'best.pt')))
print('NARRATION_CONFIG_PATH =', repr(str(RUN / 'inference_config.py')))
print('DeBERTa:', RUN / 'deberta')
print('Individual DeBERTa layers:', RUN / 'deberta_layers')
print('Individual classifier modules:', RUN / 'classifier_layers')


Best model and inference export are saved automatically at the run root.
NARRATION_MODEL_PATH = 'C:\\Users\\ESHAAN\\BYTE\\Applied-Task\\Magi\\FINETUNING-EXPERMENTS\\NarrationModelwBubbles\\model.py'
NARRATION_CHECKPOINT_PATH = 'C:\\Users\\ESHAAN\\BYTE\\Applied-Task\\Magi\\FINETUNING-EXPERMENTS\\NarrationModelwBubbles\\runs_deberta_resume_v1\\best.pt'
NARRATION_CONFIG_PATH = 'C:\\Users\\ESHAAN\\BYTE\\Applied-Task\\Magi\\FINETUNING-EXPERMENTS\\NarrationModelwBubbles\\runs_deberta_resume_v1\\inference_config.py'
DeBERTa: C:\Users\ESHAAN\BYTE\Applied-Task\Magi\FINETUNING-EXPERMENTS\NarrationModelwBubbles\runs_deberta_resume_v1\deberta
Individual DeBERTa layers: C:\Users\ESHAAN\BYTE\Applied-Task\Magi\FINETUNING-EXPERMENTS\NarrationModelwBubbles\runs_deberta_resume_v1\deberta_layers
Individual classifier modules: C:\Users\ESHAAN\BYTE\Applied-Task\Magi\FINETUNING-EXPERMENTS\NarrationModelwBubbles\runs_deberta_resume_v1\classifier_layers
